# Notebook 02 — Python Syntax Diagnostic

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 02 of 20  
**Prerequisites:** Notebook 01  
**Time estimate:** 60–90 minutes

---
## Step 1 — Motivation

This notebook is a diagnostic, not a tutorial. The purpose is to identify which Python
features are already solid (score ≥ 90% → skip) and which need deliberate practice.

Working through topics you already know wastes the 12-month budget.
Working through topics you *think* you know but can't implement wastes it more.
The diagnostic makes the difference visible.

---
## Step 2 — Intuition

Each section below is a mini-quiz: read the cell, predict the output, then run it.
If your prediction was wrong, *that topic is a gap*. If you can't write an implementation
from scratch without looking it up, *that is also a gap*.

The goal is a gap list — not a score.

---
## Step 3 — Biological Background

Not applicable to this syntax diagnostic. Biological framing begins in Notebook 04.

---
## Step 4 — Mathematical Explanation

Not applicable.

---
## Step 5 — Computational Explanation

**Topics covered in this diagnostic:**

| Topic | Why it matters in computational biology |
|-------|-----------------------------------------|
| List comprehensions | Filtering sequences, transforming expression values |
| Generators | Memory-efficient iteration over large FASTA files |
| Decorators | Caching, timing, logging in pipeline functions |
| Context managers | Safe file handling; database connections |
| Dataclasses | Typed record structures for biological data |
| Type hints | Essential for `compbio_utils` public API |
| f-strings | Formatted output in reports and logs |
| `*args`/`**kwargs` | Flexible function signatures in utility code |
| Error handling | Graceful handling of malformed biological data |

---
## Step 6 — Python Implementation

**Instructions for each section:**  
1. Predict the output *before* running.  
2. Run the cell.  
3. If you were wrong, add the topic to your gap list at the bottom of this notebook.

In [ ]:
# Section 1 — List comprehensions
# Predict the output before running.

sequences = ["ATGC", "GCGCGC", "AAATTT", "ATATATG"]

# Q1a: What does this produce?
long_seqs = [s for s in sequences if len(s) > 4]
print("Q1a:", long_seqs)

# Q1b: What does this produce?
lengths = [len(s) for s in sequences]
print("Q1b:", lengths)

# Q1c: What does this produce?
gc_flags = [s.count('G') + s.count('C') > len(s) // 2 for s in sequences]
print("Q1c:", gc_flags)

In [ ]:
# Section 2 — Generators

def fasta_reader(lines):
    """Minimal FASTA parser — yields (header, sequence) tuples."""
    header, seq = None, []
    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            if header:
                yield header, "".join(seq)
            header, seq = line[1:], []
        else:
            seq.append(line)
    if header:
        yield header, "".join(seq)

fasta_lines = [">gene1", "ATGCATGC", ">gene2", "GCGCGCGC", "ATATATAT"]

# Q2a: What type is fasta_reader(fasta_lines)?
gen = fasta_reader(fasta_lines)
print("Q2a:", type(gen))

# Q2b: What does list() do to it?
records = list(fasta_reader(fasta_lines))
print("Q2b:", records)

In [ ]:
# Section 3 — Decorators
import time, functools

def timer(func):
    """Decorator that prints execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def count_gc(sequence: str) -> int:
    return sum(1 for nt in sequence if nt in "GCgc")

# Q3a: What is printed when we call count_gc?
result = count_gc("ATGCGCATGCGC" * 1000)
print(f"GC count: {result}")

# Q3b: What is count_gc.__name__? (Without @functools.wraps, it would be different.)
print("function name:", count_gc.__name__)

In [ ]:
# Section 4 — Context managers
import io

fasta_content = ">seq1\nATGCATGC\n>seq2\nGCGCGCGC\n"
fasta_buffer = io.StringIO(fasta_content)

# Q4: Why is 'with' preferred over manual open/close?
# Write the answer as a comment, then run.
with fasta_buffer as f:
    lines = f.readlines()
print(f"Read {len(lines)} lines")

# Q4b: What happens if you try to read fasta_buffer again outside the with block?
try:
    fasta_buffer.read()
    print("Read succeeded")
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# Section 5 — Dataclasses
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class GeneRecord:
    gene_id: str
    sequence: str
    organism: str = "unknown"
    expression_values: list[float] = field(default_factory=list)

    def gc_content(self) -> float:
        if not self.sequence:
            return 0.0
        return sum(1 for nt in self.sequence if nt in "GCgc") / len(self.sequence)

gene = GeneRecord("BRCA1", "ATGCGCATGC", "Homo sapiens", [5.2, 3.1, 8.9])

# Q5a: What does print(gene) produce?
print(gene)

# Q5b: What is gene.gc_content()?
print(f"GC content: {gene.gc_content():.3f}")

# Q5c: Are two GeneRecords with the same fields equal?
gene2 = GeneRecord("BRCA1", "ATGCGCATGC", "Homo sapiens", [5.2, 3.1, 8.9])
print(f"gene == gene2: {gene == gene2}")

In [ ]:
# Section 6 — Type hints
from typing import Sequence, Union

def gc_content(sequence: str) -> float:
    """Return GC content as a fraction in [0.0, 1.0]."""
    if not sequence:
        raise ValueError("Empty sequence")
    return sum(1 for nt in sequence.upper() if nt in "GC") / len(sequence)

def mean_expression(values: Sequence[float]) -> Optional[float]:
    """Return the mean, or None if the sequence is empty."""
    return sum(values) / len(values) if values else None

print(gc_content("ATGCGCATGC"))      # Should be 0.6
print(mean_expression([5.2, 3.1, 8.9]))  # Should be ~5.73
print(mean_expression([]))               # Should be None

In [ ]:
# Section 7 — f-strings
gene_id = "BRCA1"
gc = 0.6123456
expression = [5.2, 3.1, 8.9, 4.5]

# Q7: Write these without looking up f-string syntax:
# a) "Gene: BRCA1, GC: 61.23%"
print(f"Gene: {gene_id}, GC: {gc:.2%}")

# b) "Mean expression: 5.425 (4 samples)"
print(f"Mean expression: {sum(expression)/len(expression):.3f} ({len(expression)} samples)")

# c) Left-aligned gene_id in a 15-char field
print(f"{gene_id:<15} | {gc:.4f}")

In [ ]:
# Section 8 — *args and **kwargs
def summarize(*sequences: str, separator: str = ", ") -> str:
    """Summarize multiple sequences as 'seq1, seq2, ...' with length annotations."""
    return separator.join(f"{s[:6]}..({len(s)}bp)" if len(s) > 6 else f"{s}({len(s)}bp)"
                          for s in sequences)

# Q8a:
print(summarize("ATGC", "GCGCGCGCGC", "AAATTT"))

# Q8b:
print(summarize("ATGC", "GCGCGCGCGC", separator=" | "))

# Q8c: Can you call this with a list using *unpacking?
seqs = ["ATGC", "GCGC", "TTAA"]
print(summarize(*seqs))

In [ ]:
# Section 9 — Error handling

def safe_gc_content(sequence: str) -> float:
    """GC content with graceful handling of invalid inputs."""
    if not isinstance(sequence, str):
        raise TypeError(f"Expected str, got {type(sequence).__name__}")
    sequence = sequence.upper().strip()
    valid = set("ACGTN-")
    invalid_chars = set(sequence) - valid
    if invalid_chars:
        raise ValueError(f"Invalid nucleotide characters: {invalid_chars}")
    if not sequence or all(nt in 'N-' for nt in sequence):
        return float('nan')
    countable = [nt for nt in sequence if nt not in 'N-']
    return sum(1 for nt in countable if nt in "GC") / len(countable)

test_cases = [
    ("ATGCGCATGC", None),   # valid
    ("atgcNN--",  None),    # lowercase + N + gap
    ("ATXYZ",    ValueError),  # invalid chars
    (42,          TypeError),   # wrong type
]

for seq, expected_exc in test_cases:
    try:
        result = safe_gc_content(seq)
        status = "✓" if expected_exc is None else "✗ expected exception"
        print(f"  {str(seq):<20}  gc={result:.3f}  {status}")
    except Exception as e:
        status = "✓" if type(e) is expected_exc else f"✗ got {type(e).__name__}"
        print(f"  {str(seq):<20}  {type(e).__name__}: {e}  {status}")

In [ ]:
# Section 10 — Gap list
# Fill in the topics you got wrong or couldn't predict correctly.
# Only spend time re-studying topics in this list.

gaps = [
    # Example: "Section 3 — functools.wraps: didn't know it preserves __name__",
    # Add your actual gaps here:
]

print("My gaps from this diagnostic:")
if not gaps:
    print("  (none identified — proceed to Notebook 03)")
else:
    for gap in gaps:
        print(f"  - {gap}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Score summary: rate each section 0–3
# 3 = predicted output correctly AND can write from scratch
# 2 = predicted correctly but couldn't write from scratch
# 1 = wrong prediction but understood after seeing it
# 0 = still unclear after running

scores = {
    "List comprehensions":  3,  # ← update these
    "Generators":           3,
    "Decorators":           3,
    "Context managers":     3,
    "Dataclasses":          3,
    "Type hints":           3,
    "f-strings":            3,
    "*args/**kwargs":       3,
    "Error handling":       3,
}

print("Diagnostic scores:")
for topic, score in scores.items():
    bar = "▓" * score + "░" * (3 - score)
    action = "skip" if score == 3 else ("skim" if score == 2 else "STUDY")
    print(f"  {bar}  {topic:<30} → {action}")

total = sum(scores.values())
print(f"\nTotal: {total}/{len(scores)*3}")

---
## Step 8 — Exercises

**Exercise:**  
Implement `compbio_utils.sequence.gc_content()` using **only built-in Python** (no NumPy).
The function must:
- Accept a `str` sequence
- Return a `float` in `[0.0, 1.0]`
- Raise `ValueError` for empty input
- Be case-insensitive

Write a **property-based test** using plain `pytest` (no hypothesis library) that verifies
the return value is always in `[0.0, 1.0]` for any valid uppercase DNA string.

*Solution goes in `utilities/compbio_utils/sequence.py` and `tests/test_sequence.py`.*

---
## Quiz — Active Recall

1. What is the difference between a list comprehension and a generator expression?
   When is each preferable?
2. What does `@functools.wraps(func)` do? What breaks without it?
3. What does `field(default_factory=list)` do in a dataclass? Why not `field(default=[])`?
4. What is `Optional[float]` equivalent to in Python 3.10+ syntax?
5. Why should `gc_content` raise `ValueError` on empty input rather than returning `0.0`?

---
## Reflection

**Date completed:** ____________________

> *[What was your score in Cell 7.1? Which topics need deliberate practice? Is gc_content() implemented and tested?]*

---
*Next: `03_functions_modules_packages.ipynb`*